# Notebook 05 — Feature Engineering

Build prediction-time features and create a reproducible preprocessing pipeline.

All preprocessing is fitted using the training split only and then applied unchanged to validation and test data.

In [4]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().parent

SPLITS_DIR = PROJECT_ROOT / "artifacts" / "03_splits"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "05_features"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
train_df = pd.read_parquet(SPLITS_DIR / "train.parquet")
val_df = pd.read_parquet(SPLITS_DIR / "validation.parquet")
test_df = pd.read_parquet(SPLITS_DIR / "test.parquet")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (67529, 39)
Validation: (14470, 39)
Test: (14471, 39)


## Create prediction-time features

In [6]:
def create_features(df):
    df = df.copy()

    purchase = pd.to_datetime(df["order_purchase_timestamp"])
    estimated = pd.to_datetime(df["order_estimated_delivery_date"])

    # Purchase-time features
    df["purchase_month"] = purchase.dt.month
    df["purchase_weekday"] = purchase.dt.dayofweek
    df["purchase_hour"] = purchase.dt.hour
    df["purchase_weekend"] = (purchase.dt.dayofweek >= 5).astype(int)

    # Estimated delivery window known at prediction time
    df["estimated_delivery_days"] = (
        (estimated - purchase).dt.total_seconds() / 86400
    )

    return df

In [7]:
train_fe = create_features(train_df)
val_fe = create_features(val_df)
test_fe = create_features(test_df)

## Define features and remove leakage

In [9]:
numeric_features = [
    "item_count",
    "unique_product_count",
    "unique_seller_count_x",
    "total_item_price",
    "mean_item_price",
    "max_item_price",
    "total_freight_value",
    "payment_count",
    "payment_type_count",
    "max_payment_installments",
    "total_payment_value",
    "unique_category_count",
    "unique_seller_state_count",
    "mean_product_weight_g",
    "max_product_weight_g",
    "mean_product_length_cm",
    "mean_product_height_cm",
    "mean_product_width_cm",
    "mean_seller_latitude",
    "mean_seller_longitude",
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "purchase_weekend",
    "estimated_delivery_days",
]

categorical_features = [
    "customer_state",
]

## Separate X and y

In [11]:
feature_columns = numeric_features + categorical_features

X_train = train_fe[feature_columns]
X_val = val_fe[feature_columns]
X_test = test_fe[feature_columns]

y_train = train_fe["is_late"]
y_val = val_fe["is_late"]
y_test = test_fe["is_late"]

print("X train:", X_train.shape)
print("X validation:", X_val.shape)
print("X test:", X_test.shape)

X train: (67529, 26)
X validation: (14470, 26)
X test: (14471, 26)


## Build preprocessing pipelines

In [12]:
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

In [ ]:
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

In [14]:
preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])

## Fit on training only

In [15]:
X_train_transformed = preprocessor.fit_transform(X_train)

X_val_transformed = preprocessor.transform(X_val)
X_test_transformed = preprocessor.transform(X_test)

## Get final feature names

In [17]:
feature_names = preprocessor.get_feature_names_out()

print("Number of final features:", len(feature_names))

print(feature_names[:20])

Number of final features: 52
['numeric__item_count' 'numeric__unique_product_count'
 'numeric__unique_seller_count_x' 'numeric__total_item_price'
 'numeric__mean_item_price' 'numeric__max_item_price'
 'numeric__total_freight_value' 'numeric__payment_count'
 'numeric__payment_type_count' 'numeric__max_payment_installments'
 'numeric__total_payment_value' 'numeric__unique_category_count'
 'numeric__unique_seller_state_count' 'numeric__mean_product_weight_g'
 'numeric__max_product_weight_g' 'numeric__mean_product_length_cm'
 'numeric__mean_product_height_cm' 'numeric__mean_product_width_cm'
 'numeric__mean_seller_latitude' 'numeric__mean_seller_longitude']


## Convert to feature tables & Add Target

In [18]:
X_train_final = pd.DataFrame(
    X_train_transformed,
    columns=feature_names,
    index=train_fe.index
)

X_val_final = pd.DataFrame(
    X_val_transformed,
    columns=feature_names,
    index=val_fe.index
)

X_test_final = pd.DataFrame(
    X_test_transformed,
    columns=feature_names,
    index=test_fe.index
)

In [19]:
train_final = X_train_final.copy()
val_final = X_val_final.copy()
test_final = X_test_final.copy()

train_final["is_late"] = y_train.values
val_final["is_late"] = y_val.values
test_final["is_late"] = y_test.values

## Validation

In [21]:
assert list(X_train_final.columns) == list(X_val_final.columns)
assert list(X_train_final.columns) == list(X_test_final.columns)

assert X_train_final.isna().sum().sum() == 0
assert X_val_final.isna().sum().sum() == 0
assert X_test_final.isna().sum().sum() == 0

print("Train:", train_final.shape)
print("Validation:", val_final.shape)
print("Test:", test_final.shape)

print("Missing train:", X_train_final.isna().sum().sum())
print("Missing validation:", X_val_final.isna().sum().sum())
print("Missing test:", X_test_final.isna().sum().sum())

print("Feature engineering validation passed.")

Train: (67529, 53)
Validation: (14470, 53)
Test: (14471, 53)
Missing train: 0
Missing validation: 0
Missing test: 0
Feature engineering validation passed.


## Save 

In [22]:
train_final.to_parquet(
    OUTPUT_DIR / "train_features.parquet",
    index=False
)

val_final.to_parquet(
    OUTPUT_DIR / "validation_features.parquet",
    index=False
)

test_final.to_parquet(
    OUTPUT_DIR / "test_features.parquet",
    index=False
)

In [23]:
joblib.dump(
    preprocessor,
    OUTPUT_DIR / "preprocessor.joblib"
)

['C:\\Users\\CW\\Documents\\Projects\\Dataset_Olist\\artifacts\\05_features\\preprocessor.joblib']

In [24]:
with open(
    OUTPUT_DIR / "feature_list.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        feature_names.tolist(),
        f,
        indent=2
    )

In [25]:
with open(
    OUTPUT_DIR / "input_features.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        feature_columns,
        f,
        indent=2
    )

In [26]:
for path in OUTPUT_DIR.iterdir():
    print(path.name)

feature_list.json
input_features.json
preprocessor.joblib
test_features.parquet
train_features.parquet
validation_features.parquet
